In [123]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np 
import chromadb

In [234]:
loader = PyPDFLoader("data/10201979.pdf")
loader = PyPDFLoader("data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf")
pages = loader.load()

#  persistent client => this is going to be stored locally 
chroma_client = chromadb.PersistentClient(path="deepseek_vector_db_v2")

#### text Cleaning

In [125]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    return cleaned_text

In [126]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

#### Chunking
- splitting your text information into smaller sizes

In [128]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap= 100) # created an instance of the text splitter

In [129]:
chunks = splitter.split_documents(pages)

### Create Embeddings:
- Create embeddings from our chunks
- perform a search:
    - get user input
    - calculate cosine similarity (check for similarity in text/ information)

In [132]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings

In [134]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

- user asks a question
- embedd the question 
- check for similarity in the document  

In [ ]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

## Creating VectorDB & Loading Inforamtion

In [235]:
# add data to your vector db 
collection_research = chroma_client.create_collection(name="deepseek_vector_db_v2")

# query the vector DB

In [212]:
chunks[1]

Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience Academy;http://technoscienceacademy.com', 'keywords': 'IJSRCSEIT; http:/ijsrcseit.com', 'moddate': '2026-05-05T22:07:15+05:30', 'subject': 'Research Paper', 'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology', 'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='International Journal of Scientific Research in Computer Science, Engineering and Information Technology Peer Reviewed and Refereed International Scientific Research Journal ISSN : 2456 3307')

In [214]:
len(doc_embeddings)

405

In [217]:
len(doc_embedding[0])

768

In [227]:
doc_embeddings[234][0]

[0.010645093,
 0.012411391,
 -0.006292748,
 -0.0444394,
 0.08168645,
 -0.042114474,
 0.013894102,
 -0.00089632004,
 -0.00038908655,
 -0.0064998274,
 -0.00051731465,
 -0.046210356,
 -0.022635488,
 0.036677483,
 -0.023957703,
 0.053656105,
 -0.033323176,
 -0.00054567115,
 -0.030224904,
 0.00015575039,
 -0.0016746158,
 0.0030836314,
 -0.039941497,
 -0.037792876,
 -0.13942641,
 0.038858764,
 0.047763698,
 -0.0036761179,
 0.02309143,
 0.05290319,
 -0.011759251,
 0.03548895,
 -0.004052767,
 0.03434214,
 -0.01644553,
 -0.0020942965,
 -0.03947449,
 -0.059989937,
 0.0043753907,
 0.04165023,
 0.014901045,
 0.0031610155,
 -0.0017257283,
 0.0053478824,
 0.04129195,
 0.038694862,
 0.035475824,
 -0.019101392,
 -0.011353843,
 0.01247804,
 0.012131872,
 -0.020348469,
 -0.020737054,
 0.02202564,
 0.030774338,
 -0.004239529,
 -0.021409567,
 -0.03640867,
 0.054368068,
 -0.010690971,
 -0.01969154,
 -0.07532649,
 0.006311839,
 0.03135986,
 -0.06650607,
 0.0019841879,
 0.061154414,
 -0.06750985,
 0.02137087

In [219]:
len(chunks[0].page_content)

200

In [215]:
len(chunks)

405

In [211]:
chunks[1].metadata["author"]


'TechnoScience Academy;http://technoscienceacademy.com'

In [236]:
for index in range(0,15):
    collection_research.add(
        ids = str(index),
        documents = chunks[index].page_content,
        embeddings=doc_embeddings[index][0],
        metadatas = chunks[index].metadata,
        uris = chunks[index].metadata["author"]
    )

In [241]:
# query 
response = collection_research.query(
     query_texts=["What is Traditional RAG? "], 
     query_embeddings = createEmbedding("What is Traditional RAG? "),
      n_results=4
)

In [242]:
response["documents"]

[['the information they provide is drawn from static data, making it hard for them to keep up. RAG deals with this problem by finding appropriate documents or segments from an external knowledge source',
  'Augmented Generation (RAG) works by using big language models together with external knowledge retrieval systems to increase quality and context of text it generates. Traditional LLMs are limited',
  'ws RAG to respond with more context, better facts and greater understanding. Usually, the process includes a retriever that selects the best text groups from the database related to the query,',
  'by a generator that combines the input and the selected content to produce coherent text. RAG has performed well in processes like open domain question answering, summarization and analyzing']]

In [264]:
for index in range(0,len(chunks)):
    collection_research.add(
        ids = str(index),
        documents = chunks[index].page_content,
        embeddings= doc_embeddings[index][0],
        metadatas = chunks[index].metadata ,
        uris = chunks[index].metadata["author"]
    )

In [268]:
collection_research.get(ids=["0"]) # by IDs

{'ids': ['0'],
 'embeddings': None,
 'documents': ['Copyright © 202 6 The Author(s) : This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4.0/) International Journal of Scientific Research in Computer Science,'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf',
   'creator': 'Microsoft® Word 2016',
   'page_label': '1',
   'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology',
   'keywords': 'IJSRCSEIT; http:/ijsrcseit.com',
   'page': 0,
   'subject': 'Research Paper',
   'moddate': '2026-05-05T22:07:15+05:30',
   'producer': 'Microsoft® Word 2016',
   'total_pages': 12,
   'author': 'TechnoScience Academy;http://technoscienceacademy.com',
   'creationdate': '2026-05-05T22:07:15+05:30'}]}

In [265]:
# query 
response = collection_research.query(
     query_texts=["What is Traditional RAG? "], 
     query_embeddings = createEmbedding("What is Traditional RAG? "),
      n_results=7
)

In [266]:
response.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [267]:
print(response["embeddings"])

None


In [258]:
print(response["data"])

None


In [260]:
print(response["uris"])

None


In [261]:
response["metadatas"]

[[{'page': 6,
   'moddate': '2026-05-05T22:07:15+05:30',
   'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology',
   'keywords': 'IJSRCSEIT; http:/ijsrcseit.com',
   'subject': 'Research Paper',
   'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf',
   'total_pages': 12,
   'page_label': '7',
   'producer': 'Microsoft® Word 2016',
   'creationdate': '2026-05-05T22:07:15+05:30',
   'author': 'TechnoScience Academy;http://technoscienceacademy.com',
   'creator': 'Microsoft® Word 2016'},
  {'subject': 'Research Paper',
   'moddate': '2026-05-05T22:07:15+05:30',
   'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf',
   'creationdate': '2026-05-05T22:07:15+05:30',
   'title': 'International Journal of Scientific Research in Computer Science, Engineering and Info

In [259]:
response["distances"]

[[0.7820247411727905,
  0.8483259677886963,
  0.8928294777870178,
  0.9059797525405884,
  0.9299321174621582,
  0.9387996196746826,
  0.9449593424797058]]

In [247]:
response["documents"]

[['Hindi et al. (2025) a way that RAG is used in the law. It was first explained in brief what the main RAG',
  '(RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. There is a system that gets relevant papers or information from an outside source of',
  'process RAG relies on is for a language model to use stored knowledge, obtain new information and then create a response for the user [11]. The main distinction between RAG and traditional methods in',
  'quickly become obsolete. Basically, the RAG system uses two components that work side  by side to deliver accurate, timely and contextual information [12]: 1) Retrieval System (External Knowledge',
  'for using RAG in scientific and technical question answering (QA). Section II explains the foundational concepts of RAG, focusing on its retriever and generator components. Section III explores the',
  'VI discusses conclusion and future directions. Fundamentals Study of Retrieval 

In [255]:
response["documents"][0][0]

'Hindi et al. (2025) a way that RAG is used in the law. It was first explained in brief what the main RAG'

In [254]:
response["documents"][0][-1]

'RAG. Finally, it talked about some cool things that could be researched in the future to make RAG more useful in the legal field [34]. Tural et al. (2024) emphasize that the integration of RAG'